In [2]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

In [3]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [4]:
df = pd.read_csv("/content/drive/MyDrive/Synoris Training/Day5_training/Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
df = df.drop(columns=['RowNumber','CustomerId','Surname'],axis=1)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
df['Geography'].unique()

array(['France', 'Spain', 'Germany'], dtype=object)

In [7]:
print(df['Gender'].unique())


['Female' 'Male']


## apply Encoding on Gender and Geography columns


In [8]:
label_encoder_gender = LabelEncoder()

df["Gender"] = label_encoder_gender.fit_transform(df["Gender"])
df["Gender"]

,Gender
0,0
1,0
2,0
3,0
4,0
...,...
9995,1
9996,1
9997,0
9998,1


In [9]:
df.sample()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
9132,635,France,0,33,5,0.0,2,1,0,122949.71,0


## on geography we apply OHE

In [10]:
one_hot_encoder = OneHotEncoder()
geo_encoder = one_hot_encoder.fit_transform(df[["Geography"]]).toarray()
geo_encoder

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [11]:
one_hot_encoder.categories_

[array(['France', 'Germany', 'Spain'], dtype=object)]

In [12]:
geo_df = pd.DataFrame(geo_encoder,columns=one_hot_encoder.categories_)
geo_df.head()

,France,Germany,Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


# combine ohe with data

In [13]:
df = pd.concat([df,geo_df],axis=1)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,"(France,)","(Germany,)","(Spain,)"
0,619,France,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,France,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


# save encoder in pkl file


In [14]:
import pickle

with open("label_encoder_gender.pkl","wb") as f:
  pickle.dump(label_encoder_gender,f)

with open("one_hot_encoder.pkl","wb") as f:
  pickle.dump(one_hot_encoder,f)

## devide data into dependent and independant

In [15]:
X = df.drop(columns=["Exited"],axis=1)
y = df["Exited"]

In [16]:
# splie the data
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)


In [17]:
# scaling values
scaler = preprocessing.StandardScaler()

# Drop the original 'Geography' column as it contains string values
X_train = X_train.drop(columns=['Geography'])
X_test = X_test.drop(columns=['Geography'])

X_train.columns = X_train.columns.astype(str)
X_test.columns = X_test.columns.astype(str)

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [19]:
X_test

array([[-0.57749609,  0.91324755, -0.6557859 , ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.29729735,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.52560743, -1.09499335,  0.48508334, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.81311987, -1.09499335,  0.77030065, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.41876609,  0.91324755, -0.94100321, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.24540869,  0.91324755,  0.00972116, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [20]:
with open("scaler.pkl","wb") as f:
  pickle.dump(scaler,f)

In [21]:
X_train.shape, X_test.shape

((8000, 12), (2000, 12))

# hyperperameter tunning


In [22]:
rf_params = {"max_depth": [5,8,15,10],
             "max_features":[5,7,'sqrt',8],
             "min_samples_split": [2,8,15,20],
             "n_estimators":[10,50,100,200]}

In [23]:
rf_params

{'max_depth': [5, 8, 15, 10],
 'max_features': [5, 7, 'sqrt', 8],
 'min_samples_split': [2, 8, 15, 20],
 'n_estimators': [10, 50, 100, 200]}

In [24]:
randomcv_models = [
    ('RF',RandomForestClassifier(),rf_params),
    ('XGB',XGBClassifier(),{}),
    ('LR',LogisticRegression(),{})
]

In [25]:
randomcv_models

[('RF',
  RandomForestClassifier(),
  {'max_depth': [5, 8, 15, 10],
   'max_features': [5, 7, 'sqrt', 8],
   'min_samples_split': [2, 8, 15, 20],
   'n_estimators': [10, 50, 100, 200]}),
 ('XGB',
  XGBClassifier(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bynode=None,
                colsample_bytree=None, device=None, early_stopping_rounds=None,
                enable_categorical=True, eval_metric=None, feature_types=None,
                feature_weights=None, gamma=None, grow_policy=None,
                importance_type=None, interaction_constraints=None,
                learning_rate=None, max_bin=None, max_cat_threshold=None,
                max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
                max_leaves=None, min_child_weight=None, missing=nan,
                monotone_constraints=None, multi_strategy=None, n_estimators=None,
                n_jobs=None, num_parallel_tree=None, ...),
  {}),
 ('LR', Logist

In [26]:
from sklearn.model_selection import RandomizedSearchCV

In [27]:
for name,model,params in randomcv_models:
  randomcv = RandomizedSearchCV(estimator=model,param_distributions=params,n_iter=100,cv=3,verbose=2,random_state=42,n_jobs=-1)
  randomcv.fit(X_train,y_train)
  print(f"Best parameters for {name}: {randomcv.best_params_}")

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best parameters for RF: {'n_estimators': 200, 'min_samples_split': 20, 'max_features': 5, 'max_depth': 15}
Fitting 3 folds for each of 1 candidates, totalling 3 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=100. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters for XGB: {}
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Best parameters for LR: {}


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=100. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [28]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

models = {
    "Random Forest" : RandomForestClassifier(n_estimators=200,min_samples_split=2,max_features=7,max_depth=5),
    "Xgboost":XGBClassifier(),
    "Logistic Regression":LogisticRegression()
}

for i in range(len(list(models))):
  model = list(models.values())[i]
  model.fit(X_train,y_train)

  # make predictions
  y_train_pred = model.predict(X_train)
  y_test_pred = model.predict(X_test)

  # Training set performance
  model_train_accuracy = accuracy_score(y_train,y_train_pred)
  model_train_f1 = f1_score(y_train,y_train_pred,average='weighted')
  model_train_precision = precision_score(y_train,y_train_pred)
  model_train_recall = recall_score(y_train,y_train_pred)
  model_train_roc = roc_auc_score(y_train,y_train_pred)

  # test performance
  model_test_accuracy = accuracy_score(y_test,y_test_pred)
  model_test_f1 = f1_score(y_test,y_test_pred,average='weighted')
  model_test_precision = precision_score(y_test,y_test_pred)
  model_test_recall = recall_score(y_test,y_test_pred)
  model_test_roc = roc_auc_score(y_test,y_test_pred)

  print(list(models.keys())[i])

  print('Model performance for Training set')
  print("- Accuracy: {:.4f}".format(model_train_accuracy))
  print('- F1 Score: {:.4f}'.format(model_train_f1))
  print('- Precision: {:.4f}'.format(model_train_precision))
  print('- Recall: {:.4f}'.format(model_train_recall))
  print('- ROC AUC: {:.4f}'.format(model_train_roc))

  print('----------------------------------')

  print('Model performance for Test set')
  print('- Accuracy: {:.4f}'.format(model_test_accuracy))
  print('- F1 Score: {:.4f}'.format(model_test_f1))
  print('- Precision: {:.4f}'.format(model_test_precision))
  print('- Recall: {:.4f}'.format(model_test_recall))
  print('- ROC AUC: {:.4f}'.format(model_test_roc))
  print('='*35)

Random Forest
Model performance for Training set
- Accuracy: 0.8636
- F1 Score: 0.8467
- Precision: 0.8160
- Recall: 0.4343
- ROC AUC: 0.7045
----------------------------------
Model performance for Test set
- Accuracy: 0.8640
- F1 Score: 0.8484
- Precision: 0.7738
- Recall: 0.4351
- ROC AUC: 0.7020
Xgboost
Model performance for Training set
- Accuracy: 0.9594
- F1 Score: 0.9581
- Precision: 0.9741
- Recall: 0.8242
- ROC AUC: 0.9093
----------------------------------
Model performance for Test set
- Accuracy: 0.8585
- F1 Score: 0.8490
- Precision: 0.6964
- Recall: 0.4962
- ROC AUC: 0.7216
Logistic Regression
Model performance for Training set
- Accuracy: 0.8114
- F1 Score: 0.7737
- Precision: 0.6158
- Recall: 0.2184
- ROC AUC: 0.5916
----------------------------------
Model performance for Test set
- Accuracy: 0.8110
- F1 Score: 0.7737
- Precision: 0.5524
- Recall: 0.2010
- ROC AUC: 0.5806


In [ ]:
import pickle

# Assuming the last trained model from the 'models' dictionary is the one to save
model_to_save = models["Random Forest"]

with open("random_forest_model.pkl", "wb") as f:
    pickle.dump(model_to_save, f)